# Tutoriel — Suivi des retards Rouen Rive Droite → Paris Saint-Lazare

Ce notebook déroule **toutes les étapes du pipeline**, une par une, pour comprendre le code :

1. **Configuration** — chargement des paramètres (`Settings`).
2. **Client API SNCF** — récupération des départs temps réel à Rouen Rive Droite.
3. **Stockage** — snapshots JSONL journaliers et déduplication.
4. **Flux GTFS-RT TripUpdates** — décodage protobuf sans dépendance, prédictions par gare.
5. **Détection de holds & vérification croisée** — comparaison du retard annoncé (API) avec le retard TripUpdates à Paris Saint-Lazare.
6. **Rapports** — agrégats journalier, hebdomadaire, mensuel.
7. **Génération du site** — rapports JSON + vérification croisée pour GitHub Pages.

> **Note** : la SNCF ne publie pas de positions GPS publiques. La vérification croisée utilise le flux **GTFS-RT TripUpdates** (prédictions par gare, sans token) comme seconde source indépendante.

In [1]:
import sys, os
from pathlib import Path

# Rendre le package importable depuis la racine du dépôt
ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))
sys.path.insert(0, str(ROOT / "tests"))

from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

TZ = ZoneInfo("Europe/Paris")
print("racine du dépôt :", ROOT)

racine du dépôt : /workspace/github__joubertdamien__rouenparis


## 1. Configuration

Toute la configuration passe par `rouenparis.config.Settings` : valeurs par défaut + variables d'environnement. Les gares par défaut sont Rouen Rive Droite (`stop_area:SNCF:87411017`) et Paris Saint-Lazare (`stop_area:SNCF:87271007`).

In [2]:
from rouenparis.config import Settings

settings = Settings(api_token=os.environ.get("SNCF_API_TOKEN", "<token>"))
print("origine       :", settings.origin_stop_area)
print("destination   :", settings.destination_stop_area)
print("UIC8          :", settings.origin_uic8, "→", settings.destination_uic8)
print("fenêtre       :", settings.lookahead_minutes, "min")
print("seuil retard  :", settings.min_delay_minutes, "min")
print("flux GTFS-RT  :", settings.gtfs_rt_trip_updates_url)
print("suivi TripUpdates :", settings.track_trip_updates)

origine       : stop_area:SNCF:87411017
destination   : stop_area:SNCF:87271007
UIC8          : 87411017 → 87271007
fenêtre       : 240 min
seuil retard  : 5 min
flux GTFS-RT  : https://proxy.transport.data.gouv.fr/resource/sncf-gtfs-rt-trip-updates
suivi TripUpdates : True


## 2. Client API SNCF — départs temps réel

`SncfClient.fetch_departures()` interroge `https://api.sncf.com/v1/coverage/sncf/stop_areas/<origine>/departures` avec `data_freshness=realtime`, puis pour chaque départ remonte le `vehicle_journey` pour trouver l'**arrivée à Paris Saint-Lazare** et calculer le retard à l'arrivée.

La cellule suivante utilise une **fixture enregistrée** (pas d'appel réseau). Pour un vrai appel, remplacez `FakeOpener` par rien et appelez `SncfClient(settings).fetch_departures(datetime.now(tz=TZ))` avec votre token.

In [3]:
from fixtures import FakeOpener, departures_response_for
from rouenparis.sncf_client import SncfClient

demo_day = date(2026, 10, 15)
client = SncfClient(settings, opener=FakeOpener(departures_response_for(demo_day)))
trains = client.fetch_departures(datetime(2026, 10, 15, 6, 0, tzinfo=TZ))

for train in trains:
    dep_delay = (train.departure_delay_seconds or 0) / 60
    arr_delay = (train.arrival_delay_seconds / 60) if train.arrival_delay_seconds is not None else None
    arrival = f"{train.base_arrival:%H:%M}" if train.base_arrival else "—"
    print(f"train {train.train_number:>6} ({train.line_code})  départ {train.base_departure:%H:%M} +{dep_delay:.0f} min   "
          f"arrivée prévue {arrival}   retard arrivée {arr_delay} min   statut={train.status or '—'}")

train   3123 (K22)  départ 07:12 +0 min   arrivée prévue 08:30   retard arrivée 0.0 min   statut=—
train   3125 (K22)  départ 07:30 +12 min   arrivée prévue 08:48   retard arrivée 19.0 min   statut=—
train   3401 (K24)  départ 08:05 +0 min   arrivée prévue —   retard arrivée None min   statut=cancelled


### Anatomie d'une observation

Chaque départ devient un `TrainObservation` sérialisable — c'est exactement ce qui est écrit dans les snapshots.

In [4]:
import json
print(json.dumps(trains[1].to_dict(), indent=2, ensure_ascii=False))

{
  "train_number": "3125",
  "line_code": "K22",
  "line_name": "TER 3125",
  "direction": "Paris Saint-Lazare",
  "base_departure": "2026-10-15T07:30:00+02:00",
  "realtime_departure": "2026-10-15T07:42:00+02:00",
  "departure_delay_seconds": 720,
  "departure_state": "realtime",
  "base_arrival": "2026-10-15T08:48:00+02:00",
  "realtime_arrival": "2026-10-15T09:07:00+02:00",
  "arrival_delay_seconds": 1140,
  "arrival_state": "realtime",
  "status": "",
  "data_freshness": "realtime",
  "collected_at": "2026-10-28T22:13:55.003403+01:00"
}


## 3. Stockage — snapshots et déduplication

Chaque collecte écrit une ligne JSONL dans `data/snapshots/YYYY-MM-DD.jsonl`. Comme on collecte toutes les 15 minutes, un même train est observé plusieurs fois : `dedupe_final_observations()` garde pour chaque train **la meilleure observation** (arrivée temps réel connue > retard connu > observation la plus tardive).

In [5]:
from pathlib import Path
import tempfile
from rouenparis.storage import Snapshot, append_snapshot, load_snapshots
from rouenparis.storage import dedupe_final_observations

data_dir = Path(tempfile.mkdtemp())

# Deux collectes le même jour : le matin et le soir
for hour in (7, 21):
    snapshot = Snapshot(collected_at=datetime(2026, 10, 15, hour, 30, tzinfo=TZ), trains=trains)
    append_snapshot(data_dir, snapshot)

snapshots = list(load_snapshots(data_dir))
print(f"{len(snapshots)} snapshots, {sum(len(s.trains) for s in snapshots)} observations brutes")

final = dedupe_final_observations(snapshots)
print(f"{len(final)} trains après déduplication")
for train in final:
    print(f"  train {train.train_number}: {train.departure_state} -> retard arrivée {train.arrival_delay_seconds} s")

2 snapshots, 6 observations brutes
3 trains après déduplication
  train 3123: realtime -> retard arrivée 0 s
  train 3125: realtime -> retard arrivée 1140 s
  train 3401: realtime -> retard arrivée None s


## 4. Flux GTFS-RT TripUpdates — vérification croisée des retards

La SNCF **ne publie pas de positions GPS** publiques pour ses trains. En revanche, le flux **GTFS-RT TripUpdates** (protobuf) sur [transport.data.gouv.fr](https://transport.data.gouv.fr/datasets/horaires-sncf) fournit, **sans token**, les prédictions d'arrivée/départ **par gare** pour chaque train en circulation.

Deux bonnes nouvelles :

- **aucun token requis** ;
- le décodage protobuf est fait **sans dépendance** par `rouenparis.gtfsrt_proto` (lecture manuelle du wire-format), qui décode maintenant aussi les `TripUpdate`.

La cellule suivante construit un flux GTFS-RT synthétique à la main (TripUpdate avec 3 gares) et le décode.

In [6]:
from test_trip_updates import build_feed, build_trip_update_entity, build_stop_time_update, ROUEN, VERNON, PARIS

stops = [
    build_stop_time_update(ROUEN,  departure_delay=0,   departure_time=1792041120, seq=0),
    build_stop_time_update(VERNON, arrival_delay=300,  arrival_time=1792042800, departure_delay=300, departure_time=1792042860, seq=1),
    build_stop_time_update(PARIS,  arrival_delay=600,  arrival_time=1792045120, seq=2),
]
entity = build_trip_update_entity("entity-1", "OCESN3123F1187_F:TER:FR:Line::abc", stops, timestamp=1792041120)
payload = build_feed([entity])

from rouenparis.gtfsrt_proto import parse_feed
feed = parse_feed(payload)
print(f"version={feed.version}  timestamp={feed.timestamp}")
print(f"trip_updates={len(feed.trip_updates)}  vehicles={len(feed.vehicles)}")

tu = feed.trip_updates[0]
print(f"\ntrip_id   : {tu.trip.trip_id}")
print(f"start     : {tu.trip.start_date} {tu.trip.start_time}")
print(f"stops     : {len(tu.stop_time_updates)}")
for s in tu.stop_time_updates:
    print(f"  {s.stop_id:45s} arr_delay={s.arrival.delay}  dep_delay={s.departure.delay}")

version=2.0  timestamp=1760000000
trip_updates=1  vehicles=0

trip_id   : OCESN3123F1187_F:TER:FR:Line::abc
start     : 20261028 08:00:00
stops     : 3
  StopPoint:OCETrain TER-87411017               arr_delay=None  dep_delay=0
  StopPoint:OCETrain TER-87317586               arr_delay=300  dep_delay=300
  StopPoint:OCETrain TER-87271007               arr_delay=600  dep_delay=None


### Vrai appel réseau (optionnel)

Décommentez pour interroger le vrai flux (aucun token nécessaire) :

```python
from rouenparis.trip_updates import TripUpdatesClient
profiles = TripUpdatesClient(settings).fetch_profiles()
print(len(profiles), "courses dans le flux")
```

## 5. Détection de holds et vérification croisée des retards

`rouenparis.trip_updates` transforme un profil TripUpdates en :

- **événements de hold** : quand le retard à l'arrivée à une gare dépasse le retard au départ de la gare précédente, le train a été retenu en ligne (signature d'un arrêt sans GPS) ;
- **retard à l'arrivée à Paris Saint-Lazare** : extrait indépendamment du flux, il est comparé au retard annoncé par l'API SNCF (tolérance : 60 s).

C'est exactement ce croisement que le site affiche dans la section « Vérification croisée ».

In [7]:
from datetime import datetime
from rouenparis.trip_updates import profile_from_update, detect_hold_events, cross_check_arrival_delay

collected = datetime(2026, 10, 15, 7, 12, tzinfo=TZ)
profile = profile_from_update(tu, collected)
print(f"course     : {profile.trip_id}")
print(f"n° train   : {profile.train_number}")
print(f"gares      : {len(profile.stops)}")

# Détection de holds : le retard passe de 0 à +300s à Vernon, puis +300s de plus à Paris
events = detect_hold_events(profile)
print(f"\nholds détectés : {len(events)}")
for ev in events:
    print(f"  +{ev.extra_delay_seconds}s en approchant {ev.uic8} (retard {ev.delay_before}s → {ev.delay_at}s)")

# Vérification croisée : l'API annonce +10 min (600s), TripUpdates aussi
result = cross_check_arrival_delay(profile, "87271007", announced_delay_seconds=600)
print(f"\nVérification croisée à Paris Saint-Lazare :")
print(f"  retard API SNCF   : +{result['announced_delay_seconds']}s")
print(f"  retard GTFS-RT    : +{result['trip_updates_delay_seconds']}s")
print(f"  écart             : {result['difference_seconds']}s")
print(f"  concordant        : {result['agrees']}")

course     : OCESN3123F1187_F:TER:FR:Line::abc
n° train   : 3123
gares      : 3

holds détectés : 2
  +300s en approchant 87317586 (retard 0s → 300s)
  +300s en approchant 87271007 (retard 300s → 600s)

Vérification croisée à Paris Saint-Lazare :
  retard API SNCF   : +600s
  retard GTFS-RT    : +600s
  écart             : 0s
  concordant        : True


### Rapprochement profil ↔ train

Le rapprochement se fait par **numéro commercial** (extrait du `trip_id`) + vérification que le profil dessert bien Rouen **et** Paris Saint-Lazare, avec une fenêtre de ±45 min sur le premier départ.

In [8]:
from rouenparis.trip_storage import append_trip_profiles, load_trip_profiles, cross_check_train

# On réutilise le même data_dir que la section 3 (snapshots)
append_trip_profiles(data_dir, [profile])
profiles = list(load_trip_profiles(data_dir))
print(f"{len(profiles)} profil(s) stocké(s)")

# Rapprochement + vérification croisée pour le train 3123 (départ prévu 07:12)
base_departure = datetime(2026, 10, 15, 7, 12, tzinfo=TZ)
result = cross_check_train(profiles, "3123", base_departure, "87271007", 600, origin_uic8="87411017")
print(f"\ntrain 3123 : retard API={result['announced_delay_seconds']}s  GTFS-RT={result['trip_updates_delay_seconds']}s  accord={result['agrees']}")

1 profil(s) stocké(s)

train 3123 : retard API=600s  GTFS-RT=600s  accord=True


## 6. Rapports — journalier, hebdomadaire, mensuel

`rouenparis.reports` transforme les snapshots dédupliqués en `TrainRecord`, agrège par période (jour / semaine ISO / mois) et calcule : ponctualité (seuil 5 min), retard moyen/max, suppressions — **avec ou sans** la vérification odométrique.

In [9]:
from rouenparis.reports import build_train_records, build_report, daily_series

records = build_train_records(snapshots, settings.min_delay_minutes * 60,
                              settings=settings, trip_profiles=profiles)
for record in records:
    cc = record.trip_update_cross_check
    cc_str = ""
    if cc:
        cc_str = f"  |  GTFS-RT +{cc['trip_updates_delay_seconds']}s  |  {'concordant' if cc['agrees'] else 'ECART'}"
    print(f"{record.service_date}  train {record.train_number}  prévu {record.scheduled_departure[11:16]}  "
          f"retard {record.delay_seconds} s  statut={record.status}{cc_str}")

daily = build_report(records, "daily", date(2026, 10, 15), settings)
print("\nRésumé du 2026-10-15 :", json.dumps(daily["summary"], ensure_ascii=False))

2026-10-15  train 3123  prévu 07:12  retard 0 s  statut=scheduled  |  GTFS-RT +600s  |  ECART
2026-10-15  train 3125  prévu 07:30  retard 1140 s  statut=scheduled
2026-10-15  train 3401  prévu 08:05  retard 0 s  statut=cancelled

Résumé du 2026-10-15 : {"scheduled": 3, "circulated": 2, "cancelled": 1, "measured": 2, "delayed": 1, "punctuality_rate": 50.0, "avg_delay_minutes": 9.5, "max_delay_minutes": 19.0, "min_delay_threshold_minutes": 5, "cross_checked": 1, "cross_check_agreeing": 0}


In [10]:
weekly = build_report(records, "weekly", date(2026, 10, 12), settings)
monthly = build_report(records, "monthly", date(2026, 10, 1), settings)
print("semaine  :", weekly["period"], "->", weekly["summary"]["punctuality_rate"], "% de ponctualité")
print("mois     :", monthly["period"], "->", monthly["summary"]["punctuality_rate"], "% de ponctualité")

for entry in daily_series(records, settings.min_delay_minutes * 60):
    print(f"série {entry['date']}: ponctualité={entry['punctuality_rate']}%  retard moyen={entry['avg_delay_minutes']} min")

semaine  : {'type': 'weekly', 'start': '2026-10-12', 'end': '2026-10-18'} -> 50.0 % de ponctualité
mois     : {'type': 'monthly', 'start': '2026-10-01', 'end': '2026-10-31'} -> 50.0 % de ponctualité
série 2026-10-15: ponctualité=50.0%  retard moyen=9.5 min


## 7. Génération du site (GitHub Pages)

`generate_all()` écrit dans `site-build/` :

- `index.json` — liste des rapports J/W/M + bloc odométrie ;
- `series.json` — série temporelle de ponctualité journalière ;
- `odometry.json` — par course : fixes GPS, distance, arrêts détectés ;
- `reports/daily-*.json`, `reports/weekly-*.json`, `reports/monthly-*.json` — rapports complets avec, pour chaque train, le retard annoncé **et** le retard estimé par odométrie.

Le workflow `pages.yml` copie ensuite `site/` (HTML/CSS/JS statiques) dans ce répertoire et publie le tout sur GitHub Pages.

In [11]:
from rouenparis.reports import generate_all

site_dir = Path(tempfile.mkdtemp()) / "site-build"
index = generate_all(data_dir, site_dir, settings)
print("fichiers générés :")
for p in sorted(site_dir.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(site_dir)}  ({p.stat().st_size} octets)")

fichiers générés :
  crosscheck.json  (662 octets)
  index.json  (2205 octets)
  latest.json  (87 octets)
  odometry.json  (217 octets)
  reports/daily-2026-10-15.json  (2902 octets)
  reports/monthly-2026-10-01.json  (2904 octets)
  reports/weekly-2026-10-12.json  (2903 octets)
  series.json  (453 octets)


In [12]:
report = json.loads((site_dir / "reports" / "daily-2026-10-15.json").read_text(encoding="utf-8"))
print(f"{report['period']['type']} {report['period']['start']} → {report['summary']['scheduled']} trains, ponctualité {report['summary']['punctuality_rate']}%")
for train in report["trains"]:
    tu = train.get("trip_update_delay_minutes")
    agree = train.get("delay_cross_check_agrees")
    print(f"  train {train['train_number']}: retard {train['delay_minutes']:+.1f} min"
          + (f"  |  GTFS-RT {tu:+.1f} min  |  {'✅' if agree else '❌'}" if tu is not None else ""))

# La vérification croisée globale
crosscheck = json.loads((site_dir / "crosscheck.json").read_text(encoding="utf-8"))
print(f"\nVérification croisée : {crosscheck['trains_cross_checked']} trains vérifiés, "
      f"{crosscheck['delays_agreeing']} concordants")

daily 2026-10-15 → 3 trains, ponctualité 50.0%
  train 3123: retard +0.0 min  |  GTFS-RT +10.0 min  |  ❌
  train 3125: retard +19.0 min
  train 3401: retard +0.0 min

Vérification croisée : 1 trains vérifiés, 0 concordants


## 8. Récapitulatif — commandes équivalentes

| Étape du notebook | Commande CLI |
|---|---|
| 1. Configuration | `python -m rouenparis resolve-stations` |
| 2. Client API SNCF | `python -m rouenparis collect --count 1` |
| 3. Stockage | *(automatique dans `collect`)* |
| 4. Flux GTFS-RT TripUpdates | `python -m rouenparis collect-trip-updates` |
| 5. Vérification croisée | *(automatique dans `report`)* |
| 6. Rapports J/W/M | `python -m rouenparis report` |
| 7. Génération du site | *(inclus dans `report`)* |

Tout le pipeline en une commande : `python -m rouenparis all`